In [1]:
import os
import random
import sys

import math
import numpy as np
import torch

from torch import nn
from pathlib import Path

sys.path.append(os.path.abspath('./'))

SEED = 12
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

from input_image_gen import generate_gemm_input_file, generate_gemm_int_input_file
from codebooks_defs_generator import (
    generate_cb_definitions,
    build_generated_registry_header,
    write_generated_registry_header
)

from gemm_layer_generator import (
    export_ticsat_weight_file,
    generate_gemm_data_file,
    generate_template_gemm,
    _reverse_input_groups_of_4_rows,
    blockwise_to_rowwise,
    unpack_rowwise_words_to_matrix
)

from transformer_debug_utils import (
    compare_saved_outputs_with_reference,
    compare_saved_float_outputs_with_reference,
    export_transformer_float_outputs,
    print_float_comparison_summary,
    transformer_block_forward_fp32,
    c_output_dir_has_any_outputs,
    export_transformer_outputs,
    print_comparison_summary,
    transformer_block_forward_c_style,
)


* This notebook generates **TiC-SAT-ready artifacts for the complete Transformer GEMM pipeline**
* It produces the shared definition headers, including `codebooks_def.h` and `input_matrix.h`, for integration with the TiC-SAT codebase. 
* It generates one `gemm_header_<id>.h` file for each GEMM layer in the Transformer block, covering:

  * the **Q/K/V projection layers** for each attention head,
  * the **post-attention condense/projection layer**, and
  * the **two feed-forward network (FFN) layers**. 
* It writes `generated_codebook_registry.h` to register and organize the generated codebook metadata for all exported layers. 
* It exports **TiC-SAT-compatible packed binary weight files** for the same set of layers into the `weights/generated_from_notebook` directory. 
* It also performs a **PyTorch-based end-to-end debug forward pass** to validate the correctness of the generated Transformer block outputs. 

In [2]:
""" TiC-SAT Transformer Hyperparameters"""
# Menu Templates
# TRANSFORMER_D_Q = 
# TRANSFORMER_SEQ_LEN = 
# TRANSFORMER_D_MODEL = 
# TRANSFORMER_NUM_HEADS = 
# TRANSFORMER_D_FF = 


# Extra Small Menu
# TRANSFORMER_SEQ_LEN = 2
# TRANSFORMER_D_MODEL = 8
# TRANSFORMER_D_FF = 4

# Smaller Smaller Menu
# TRANSFORMER_D_Q = 4
# TRANSFORMER_SEQ_LEN = 2
# TRANSFORMER_D_MODEL = 8
# TRANSFORMER_NUM_HEADS = 2
# TRANSFORMER_D_FF = 8

# Smaller Menu
# TRANSFORMER_D_Q = 4
# TRANSFORMER_SEQ_LEN = 4
# TRANSFORMER_D_MODEL = 8
# TRANSFORMER_NUM_HEADS = 2
# TRANSFORMER_D_FF = 4

# Small Menu
# TRANSFORMER_SEQ_LEN = 8
# TRANSFORMER_D_MODEL = 8
# TRANSFORMER_D_FF = 16

# Medium Menu
# TRANSFORMER_SEQ_LEN = 8
# TRANSFORMER_D_MODEL = 16
# TRANSFORMER_D_FF = 32

# Large Menu 
# TRANSFORMER_SEQ_LEN = 16
# TRANSFORMER_D_MODEL = 32
# TRANSFORMER_D_FF = 4

# Advanced Menu 1
# TRANSFORMER_SEQ_LEN = 32
# TRANSFORMER_D_MODEL = 64
# TRANSFORMER_D_FF = 128

# Advanced Menu 2
# TRANSFORMER_D_Q = 16
# TRANSFORMER_SEQ_LEN = 64
# TRANSFORMER_D_MODEL = 128
# TRANSFORMER_NUM_HEADS = 8
# TRANSFORMER_D_FF = 256

# BERT-mini
TRANSFORMER_D_Q = 64
TRANSFORMER_SEQ_LEN = 512
TRANSFORMER_D_MODEL = 256
TRANSFORMER_NUM_HEADS = 4
TRANSFORMER_D_FF = 1024

# Boss Menu (BERT-base)
# TRANSFORMER_D_Q = 64
# TRANSFORMER_SEQ_LEN = 512
# TRANSFORMER_D_MODEL = 768
# TRANSFORMER_NUM_HEADS = 12
# TRANSFORMER_D_FF = 3072


TRANSFORMER_HEAD_HIDDEN_SIZE = TRANSFORMER_D_MODEL // TRANSFORMER_NUM_HEADS
CONDENSE_IN_SIZE = TRANSFORMER_NUM_HEADS * TRANSFORMER_HEAD_HIDDEN_SIZE


""" Set weight export path for TiC-SAT """

TICSAT_REPO_ROOT = '/home/thu/TiC-SAT'
GENERATE_TICSAT_BINS = True

TICSAT_SA_SIZE = 4 # Change this if SA_SIZE in TiC-SAT is changed!!!
TICSAT_KERNEL_DIM = TICSAT_SA_SIZE # SA_SIZE 
TICSAT_QUANT_SCALE = 32.0 # This is the scale factor we will use to quantize our weights to int8. Not required to change this if you are using the default quantization scheme in TiC-SAT which is to divide by 32 and round to nearest int.
TICSAT_OUT_FOLDER = os.path.join(TICSAT_REPO_ROOT, 'weights', 'generated_from_notebook') # Change this if you want to export to a different folder in the TiC-SAT repo
os.makedirs(TICSAT_OUT_FOLDER, exist_ok=True)

OUT_FOLDER = os.path.join(TICSAT_REPO_ROOT, 'Full_NN', 'gemm_definitions') + '/'
os.makedirs(OUT_FOLDER, exist_ok=True)

In [3]:
""" Generator Hyperparameters """
N_LEARNERS = 4
CODEBOOK_SIZE = 8 # CODEBOOK_SIZE = 4, 8, 16 for one learner have been tested and results match the default dense results.
USE_CODEBOOKS = True # Always set to True in this project
SVE_LANES = 4 # The size of SVE register / 32 bits
SAME_SEQ = True # For shared index implementation, set SAME_SEQ = True. For independent sequence implementation, set SAME_SEQ = False. 
USE_FP32_TRANSFORMER = False # If True, the generator will generate a FP32 version of the transformer block in addition to the int8 version.


### Not used in current version ###
USE_BIAS = False
USE_F16 = False
TILE_L1_SIZE = 1 # 0 or 1 disables GEMM L1 tiling
TILE_L2_SIZE = 1 # 0 or 1 disables GEMM L2 tiling
# TILE_SIZE = TILE_L1_SIZE # Legacy alias used by older generator helpers
###################################

D_Q = TRANSFORMER_D_Q
SEQ_LEN = TRANSFORMER_SEQ_LEN
D_MODEL = TRANSFORMER_D_MODEL
NUM_HEAD = TRANSFORMER_NUM_HEADS
D_FF = TRANSFORMER_D_FF

# INPUT_SIZE = TRANSFORMER_D_MODEL
INPUT_SIZE = CONDENSE_IN_SIZE # The same as D_MODEL


In [4]:
''' 
In this function we build a list of dictionaries where each dictionary contains the information for one GEMM layer in the transformer block. 
This includes the name of the layer, the input and output sizes, the path to the corresponding TiC-SAT binary file.
'''
def build_gemm_layers(num_head, d_model, d_q, d_ff):
    layers = []

    for h in range(num_head):
        layers.append({
            "name": f"q_h{h}",
            "type": "gemm",
            "input_size": d_model,
            "output_size": d_q,
            "ticsat_bin": f"H{h}_L0.bin",
            "fallback_weight_index": h * 3 + 0,
        })
        layers.append({
            "name": f"k_h{h}",
            "type": "gemm",
            "input_size": d_model,
            "output_size": d_q,
            "ticsat_bin": f"H{h}_L1.bin",
            "fallback_weight_index": h * 3 + 1,
        })
        layers.append({
            "name": f"v_h{h}",
            "type": "gemm",
            "input_size": d_model,
            "output_size": d_q,
            "ticsat_bin": f"H{h}_L2.bin",
            "fallback_weight_index": h * 3 + 2,
        })

    layers.extend([
        {
            "name": "condense",
            "type": "gemm",
            "input_size": num_head * d_q,
            "output_size": d_model,
            "ticsat_bin": "H-1_L0.bin",
            "fallback_weight_index": num_head * 3 + 0,
        },
        {
            "name": "ff0",
            "type": "gemm",
            "input_size": d_model,
            "output_size": d_ff,
            "ticsat_bin": "H-1_L1.bin",
            "fallback_weight_index": num_head * 3 + 1,
        },
        {
            "name": "ff1",
            "type": "gemm",
            "input_size": d_ff,
            "output_size": d_model,
            "ticsat_bin": "H-1_L2.bin",
            "fallback_weight_index": num_head * 3 + 2,
        },
    ])

    return layers


GEMM_LAYERS = build_gemm_layers(NUM_HEAD, D_MODEL, D_Q, D_FF)

for layer in GEMM_LAYERS:
    print(layer["name"], layer["input_size"], layer["output_size"], layer["ticsat_bin"])

q_h0 256 64 H0_L0.bin
k_h0 256 64 H0_L1.bin
v_h0 256 64 H0_L2.bin
q_h1 256 64 H1_L0.bin
k_h1 256 64 H1_L1.bin
v_h1 256 64 H1_L2.bin
q_h2 256 64 H2_L0.bin
k_h2 256 64 H2_L1.bin
v_h2 256 64 H2_L2.bin
q_h3 256 64 H3_L0.bin
k_h3 256 64 H3_L1.bin
v_h3 256 64 H3_L2.bin
condense 256 256 H-1_L0.bin
ff0 256 1024 H-1_L1.bin
ff1 1024 256 H-1_L2.bin


In [5]:
"""
Transformer GEMM artifact generator for TiC-SAT. This notebook generates the complete set of C/C++ headers, input data, packed weights, and debug references needed to run a Transformer block through the TiC-SAT GEMM pipeline.

It produces several related outputs:

1. A codebooks_def.h-style configuration header. This defines global compile-time constants such as the number of learners, codebook size, SVE vector layout, shared-index mode, bias/fp16 flags, and GEMM tiling settings.

2. An input_matrix.h-style input header. This contains the generated Transformer input matrix, either in fp32 form or in the int8 format used by the TiC-SAT integer pipeline.

3. One gemm_header_<id>.h file per Transformer GEMM layer. The generated layers include the Q/K/V projection GEMMs for every attention head, the post-attention condense/projection GEMM, and the two feed-forward network GEMMs.

4. A generated_codebook_registry.h-style registry header. This collects metadata for all generated GEMM/codebook layers so the generated TiC-SAT kernels can look up layer-specific packed indexes, codebooks, dimensions, and names.

5. TiC-SAT-compatible binary weight files under weights/generated_from_notebook. These files are exported per learner and per layer using the blockwise packed layout expected by the TiC-SAT hardware/software kernels.

The notebook also builds matching PyTorch nn.Linear layers from the generated weights and runs a debug Transformer forward pass. 
It exports Python reference outputs and can compare them against dumped C/TiC-SAT outputs layer by layer, covering multi-head attention output, condense output, residual/addnorm points, FFN outputs, and final Transformer output. 
This makes the notebook both a generator and a correctness-checking tool for the generated Transformer GEMM pipeline.
"""


generate_cb_definitions(
    OUT_FOLDER + 'codebooks_def.h',
    N_LEARNERS,
    CODEBOOK_SIZE,
    SVE_LANES,
    USE_BIAS,
    USE_F16,
    SAME_SEQ,
    USE_CODEBOOKS,
    tile_l1_size=TILE_L1_SIZE,
    tile_l2_size=TILE_L2_SIZE,
)

if USE_FP32_TRANSFORMER:
    input_matrix = generate_gemm_input_file(
        OUT_FOLDER + 'input_matrix.h',
        SEQ_LEN,
        INPUT_SIZE,
        USE_F16,
    )
else:
    input_matrix = generate_gemm_int_input_file(
        OUT_FOLDER + 'input_matrix.h',
        SEQ_LEN,
        INPUT_SIZE,
        USE_F16,
        ticsat_export_path=os.path.join(TICSAT_OUT_FOLDER, 'H-1_L-1.bin'),
        legacy_export_path=os.path.join(TICSAT_REPO_ROOT, 'weights', 'H-1_L-1.bin'),
        ticsat_kernel_dim=TICSAT_KERNEL_DIM,
    )

""" Test input matrix for debugging:"""
# input_matrix = [
#     [0, 0, 0, -1, 0, 0, 0, 0],
#     [0, -1, 0, 0, -1, 0, 0, 0]
# ]



network = [nn.ModuleDict({}) for _ in range(N_LEARNERS)]

generated_registry_entries = [] # To keep track of generated registry entries for all layers, for debugging and verification purposes

# in_size = INPUT_SIZE
gemm_values = []
gemm_biases = []
ticsat_exports = []

# for lay_cnt, layer in enumerate(GEMM_STRUCTURE):
for lay_cnt, layer in enumerate(GEMM_LAYERS):
    print('[{}] {}'.format(lay_cnt, layer['type']))
    print('	', layer)

    if layer['type'] != 'gemm':
        print('ERROR! Unsupported layer type:', layer['type'])
        raise SystemExit(1)


    # Use explicit per-layer dimensions instead of chaining them serially
    cur_in_size = layer["input_size"]
    cur_out_size = layer["output_size"]
    
    in_shape = (SEQ_LEN, cur_in_size)
    out_shape = (SEQ_LEN, cur_out_size)


    layer_values, layer_biases, registry_meta = generate_template_gemm(
        SAME_SEQ,
        OUT_FOLDER + 'gemm_header_{}.h'.format(lay_cnt),
        lay_cnt,
        layer['name'],
        N_LEARNERS,
        CODEBOOK_SIZE,
        TILE_L1_SIZE,
        cur_in_size,
        cur_out_size,
        USE_F16,
        USE_CODEBOOKS,
        USE_BIAS,
        use_fp32_transformer=USE_FP32_TRANSFORMER,
    )

    print("Check layer_values", layer_values)

    gemm_values.append(layer_values)
    gemm_biases.append(layer_biases)

    if registry_meta is not None:
        generated_registry_entries.append(registry_meta)

    # Export weights for TiC-SAT if needed, and check correctness
    if GENERATE_TICSAT_BINS and not USE_FP32_TRANSFORMER and 'ticsat_bin' in layer:
        for learner in range(N_LEARNERS):
            bin_name = layer['ticsat_bin']
            learner_dir = os.path.join(TICSAT_OUT_FOLDER, f'learner{learner}')
            export_path = os.path.join(learner_dir, bin_name)
            

            blockwise_words = export_ticsat_weight_file(
                export_path,
                layer_values[learner],
                cur_in_size,
                cur_out_size,
                kernel_dim=TICSAT_KERNEL_DIM,
                quant_scale=TICSAT_QUANT_SCALE,
            )   

            ticsat_exports.append(export_path)


            # ===== Check the correctness of exported blockwise weights =====
            weights_arr = np.asarray(layer_values[learner], dtype=np.int8).reshape(cur_out_size, cur_in_size)

            # Expected RWMA rowwise matrix: [in, out]
            expected_rowwise = weights_arr.T
            expected_rowwise = _reverse_input_groups_of_4_rows(expected_rowwise)


            rowwise_words = blockwise_to_rowwise(
                blockwise_words,
                cur_in_size,
                cur_out_size,
                TICSAT_KERNEL_DIM
            )
            decoded = unpack_rowwise_words_to_matrix(
                rowwise_words,
                cur_in_size,
                cur_out_size
            )

            ok = np.array_equal(decoded, expected_rowwise)

            print(f'RWMA export check | layer={layer["name"]} learner={learner} | match={ok}')

            if not ok:
                print('Expected RWMA rowwise matrix:')
                print(expected_rowwise)
                print('Decoded from exported .bin:')
                print(decoded)

                diff_idx = np.argwhere(decoded != expected_rowwise)
                print('First mismatches:')
                for k, (r, c) in enumerate(diff_idx[:10]):
                    print(
                        f'  ({r}, {c}) expected={expected_rowwise[r, c]} decoded={decoded[r, c]}'
                    )



    # ===== Create nn.Linear layer with the generated weights, and add to network =====
    for learner in range(N_LEARNERS):
        linear = nn.Linear(cur_in_size, cur_out_size, bias=USE_BIAS)
        # weight = torch.tensor(np.asarray(layer_values[learner]).reshape(out_size, in_size), dtype=torch.int8)
        weight = torch.tensor(np.asarray(layer_values[learner]).reshape(cur_out_size, cur_in_size), dtype=torch.float32)
        with torch.no_grad():
            linear.weight.copy_(weight)
            if USE_BIAS:
                linear.bias.copy_(torch.tensor(layer_biases[learner], dtype=torch.float32))
        network[learner][layer['name']] = linear

    print('In shape:', in_shape)
    print('Out shape:', out_shape)
    print()


# generate_gemm_data_file(OUT_FOLDER + 'gemm_data.h', len(GEMM_STRUCTURE))
# generate_gemm_data_file(OUT_FOLDER + 'gemm_data.h', len(GEMM_LAYERS)) # Require to be removed, if not using GEMM_LAYERS as the source of truth for layer definitions

write_generated_registry_header(
    generated_registry_entries,
    OUT_FOLDER + 'generated_codebook_registry.h'
)

print('Generated files in', OUT_FOLDER)
print('Input matrix:')
print(input_matrix)
if ticsat_exports:
    print('TiC-SAT weight exports:')
    for path in ticsat_exports:
        print(' -', path)


[0] gemm
	 {'name': 'q_h0', 'type': 'gemm', 'input_size': 256, 'output_size': 64, 'ticsat_bin': 'H0_L0.bin', 'fallback_weight_index': 0}
Check layer_values [array([ 0, -2,  0, ...,  1, -1,  0], dtype=int8), array([-2,  0,  0, ..., -1, -2, -2], dtype=int8), array([ 1, -1, -2, ..., -1, -1,  1], dtype=int8), array([ 1,  0, -1, ...,  0,  0,  1], dtype=int8)]
RWMA export check | layer=q_h0 learner=0 | match=True
RWMA export check | layer=q_h0 learner=1 | match=True
RWMA export check | layer=q_h0 learner=2 | match=True
RWMA export check | layer=q_h0 learner=3 | match=True
In shape: (512, 256)
Out shape: (512, 64)

[1] gemm
	 {'name': 'k_h0', 'type': 'gemm', 'input_size': 256, 'output_size': 64, 'ticsat_bin': 'H0_L1.bin', 'fallback_weight_index': 1}
Check layer_values [array([-2, -1,  1, ..., -1, -2, -2], dtype=int8), array([-2, -2,  1, ..., -2, -1, -1], dtype=int8), array([-1, -1, -2, ..., -1,  0,  0], dtype=int8), array([-2, -2, -1, ..., -2, -1, -1], dtype=int8)]
RWMA export check | layer=k

In [6]:
""" Print out the generated GEMM layers and their mapping to TiC-SAT kernels, for verification and debugging purposes."""

print("SA_SIZE =", TICSAT_KERNEL_DIM)
print("TICSAT_KERNEL_DIM =", TICSAT_KERNEL_DIM)
print("MAX_COL =", TICSAT_KERNEL_DIM // 4)
print()

print("=== Transformer GEMM structure ===")
print(f"NUM_HEAD = {NUM_HEAD}")
print(f"D_MODEL  = {D_MODEL}")
print(f"D_Q      = {D_Q}")
print(f"D_FF     = {D_FF}")
print()

for h in range(NUM_HEAD):
    print(f"[Head {h}]")
    print(f"  q_h{h}: [{D_MODEL} x {D_Q}]")
    print(f"  k_h{h}: [{D_MODEL} x {D_Q}]")
    print(f"  v_h{h}: [{D_MODEL} x {D_Q}]")
    print()

print("[Post-attention]")
print(f"  condense: [{NUM_HEAD * D_Q} x {D_MODEL}]")
print()

print("[Feed-forward]")
print(f"  ff0: [{D_MODEL} x {D_FF}]")
print(f"  ff1: [{D_FF} x {D_MODEL}]")
print()


def print_layer_mapping(layer):
    in_size = layer["input_size"]
    out_size = layer["output_size"]
    packed_cols = out_size // 4
    max_col = TICSAT_KERNEL_DIM // 4

    print(f'{layer["name"]}: [{in_size} x {out_size}]')
    print(f"  packed_cols = {packed_cols}")
    print(f"  MAX_COL = {max_col}")
    print(f"  packed_cols // MAX_COL = {packed_cols // max_col}")
    print()

print("=== Transformer GEMM structure + TiC-SAT mapping ===")
for layer in GEMM_LAYERS:
    print_layer_mapping(layer)

SA_SIZE = 4
TICSAT_KERNEL_DIM = 4
MAX_COL = 1

=== Transformer GEMM structure ===
NUM_HEAD = 4
D_MODEL  = 256
D_Q      = 64
D_FF     = 1024

[Head 0]
  q_h0: [256 x 64]
  k_h0: [256 x 64]
  v_h0: [256 x 64]

[Head 1]
  q_h1: [256 x 64]
  k_h1: [256 x 64]
  v_h1: [256 x 64]

[Head 2]
  q_h2: [256 x 64]
  k_h2: [256 x 64]
  v_h2: [256 x 64]

[Head 3]
  q_h3: [256 x 64]
  k_h3: [256 x 64]
  v_h3: [256 x 64]

[Post-attention]
  condense: [256 x 256]

[Feed-forward]
  ff0: [256 x 1024]
  ff1: [1024 x 256]

=== Transformer GEMM structure + TiC-SAT mapping ===
q_h0: [256 x 64]
  packed_cols = 16
  MAX_COL = 1
  packed_cols // MAX_COL = 16

k_h0: [256 x 64]
  packed_cols = 16
  MAX_COL = 1
  packed_cols // MAX_COL = 16

v_h0: [256 x 64]
  packed_cols = 16
  MAX_COL = 1
  packed_cols // MAX_COL = 16

q_h1: [256 x 64]
  packed_cols = 16
  MAX_COL = 1
  packed_cols // MAX_COL = 16

k_h1: [256 x 64]
  packed_cols = 16
  MAX_COL = 1
  packed_cols // MAX_COL = 16

v_h1: [256 x 64]
  packed_cols = 16

In [7]:
def transformer_block_forward_debug(network_one_learner, x, num_head, d_q):
    """
    Debug-only Transformer block forward. USE_FP32_TRANSFORMER selects the
    true float32 path; otherwise the existing TiC-SAT int8-style path is kept.
    """
    if USE_FP32_TRANSFORMER:
        return transformer_block_forward_fp32(
            network_one_learner=network_one_learner,
            x=x,
            num_head=num_head,
            d_q=d_q,
        )

    return transformer_block_forward_c_style(
        network_one_learner=network_one_learner,
        x=x,
        num_head=num_head,
        d_q=d_q,
    )


In [8]:
"""
Layer by layer comparison
Only after running C transformer and dumping outputs .txt, then we can use the following code to compare the outputs layer by layer, and print out the comparison results. 
This is useful for debugging and verifying the correctness of the C transformer implementation, by comparing it with the Python reference implementation.
"""

def as_int32_numpy(x):
    if isinstance(x, torch.Tensor):
        return x.detach().cpu().to(torch.int32).numpy()
    return np.asarray(x, dtype=np.int32)



MULTI_OUTPUT_ROOT = os.path.join(TICSAT_REPO_ROOT, 'weights', 'multiple_learner_outputs')
PYTHON_OUTPUT_ROOT = os.path.join(MULTI_OUTPUT_ROOT, 'python')
C_OUTPUT_ROOT = os.path.join(MULTI_OUTPUT_ROOT, 'c')

input_tensor = np.asarray(input_matrix, dtype=np.float32 if USE_FP32_TRANSFORMER else np.int8)
print('input shape:', input_tensor.shape)

os.makedirs(PYTHON_OUTPUT_ROOT, exist_ok=True)

has_c_outputs = c_output_dir_has_any_outputs(C_OUTPUT_ROOT, N_LEARNERS)
if not has_c_outputs:
    print(f'C output directory not ready under {C_OUTPUT_ROOT}; notebook will export Python reference only.')

for ens in range(N_LEARNERS):
    print(f'\n=============== LEARNER {ens} ===============\n')

    outputs = transformer_block_forward_debug(
        network_one_learner=network[ens],
        x=input_tensor,
        num_head=NUM_HEAD,
        d_q=D_Q,
    )

    learner_python_dir = os.path.join(PYTHON_OUTPUT_ROOT, f'learner{ens}')
    if USE_FP32_TRANSFORMER:
        export_transformer_float_outputs(outputs, learner_python_dir)
    else:
        export_transformer_outputs(outputs, learner_python_dir)
    print(f'Exported Python reference outputs to: {learner_python_dir}')

    for key in [
        'multihead_out',
        'condense_out',
        'after_attn_addnorm',
        'ff0_out',
        'ff1_out',
        'final_out',
    ]:
        # print(f'\n{key} shape:', outputs[key].shape)
        # print(outputs[key].astype(np.int32))
        arr = as_int32_numpy(outputs[key])
        print(f'\n{key} shape:', arr.shape)
        # print(arr)

    if has_c_outputs:
        learner_c_dir = os.path.join(C_OUTPUT_ROOT, f'learner{ens}')
        if USE_FP32_TRANSFORMER:
            comparisons = compare_saved_float_outputs_with_reference(outputs, learner_c_dir)
            print_float_comparison_summary(ens, comparisons)
        else:
            comparisons = compare_saved_outputs_with_reference(outputs, learner_c_dir)
            print_comparison_summary(ens, comparisons)


input shape: (512, 256)

=============== LEARNER 0 ===============

Exported Python reference outputs to: /home/thu/TiC-SAT/weights/multiple_learner_outputs/python/learner0

multihead_out shape: (512, 256)

condense_out shape: (512, 256)

after_attn_addnorm shape: (512, 256)

ff0_out shape: (512, 1024)

ff1_out shape: (512, 256)

final_out shape: (512, 256)

===== C vs Python comparison | learner 0 =====
q_h0: max_abs_diff=170, mismatches=32471/32768
  first mismatch at (0, 0): python=-23 c=-25
k_h0: max_abs_diff=217, mismatches=32521/32768
  first mismatch at (0, 0): python=-26 c=48
v_h0: max_abs_diff=187, mismatches=32538/32768
  first mismatch at (0, 0): python=13 c=9
head_out_h0: max_abs_diff=255, mismatches=32661/32768
  first mismatch at (0, 0): python=80 c=103
q_h1: max_abs_diff=192, mismatches=32522/32768
  first mismatch at (0, 0): python=10 c=-74
k_h1: max_abs_diff=213, mismatches=32549/32768
  first mismatch at (0, 0): python=42 c=27
v_h1: max_abs_diff=181, mismatches=32549/